# Getting Started with Ollama Langchain
This link discusses downloading ollama, pulling a model, then running it on windows or linux

https://python.langchain.com/docs/integrations/chat/ollama/

In [ ]:
conda install -c conda-forge numpy=1.26.4 faiss-cpu=1.8.0 -y

In [1]:
!python --version

Python 3.9.20


In [2]:
!pip install \
    fastapi==0.115.6 \
    langchain==0.3.13 \
    langchain-community==0.3.12 \
    langchain-core==0.3.27 \
    langchain-nvidia-ai-endpoints==0.3.7 \
    sentence-transformers==3.3.1 \
    unstructured==0.16.11

Looking in indexes: https://pypi.org/simple, https://pypi.ngc.nvidia.com


ERROR: Ignored the following versions that require a different python version: 0.3.0 Requires-Python <4.0,>=3.9; 0.3.0.dev1 Requires-Python <4.0,>=3.9; 0.3.0.dev2 Requires-Python <4.0,>=3.9; 0.3.1 Requires-Python <4.0,>=3.9; 0.3.10 Requires-Python <4.0,>=3.9; 0.3.11 Requires-Python <4.0,>=3.9; 0.3.12 Requires-Python <4.0,>=3.9; 0.3.13 Requires-Python <4.0,>=3.9; 0.3.14 Requires-Python <4.0,>=3.9; 0.3.15 Requires-Python <4.0,>=3.9; 0.3.16 Requires-Python <4.0,>=3.9; 0.3.2 Requires-Python <4.0,>=3.9; 0.3.3 Requires-Python <4.0,>=3.9; 0.3.4 Requires-Python <4.0,>=3.9; 0.3.5 Requires-Python <4.0,>=3.9; 0.3.6 Requires-Python <4.0,>=3.9; 0.3.7 Requires-Python <4.0,>=3.9; 0.3.8 Requires-Python <4.0,>=3.9; 0.3.9 Requires-Python <4.0,>=3.9
ERROR: Could not find a version that satisfies the requirement langchain==0.3.13 (from versions: 0.0.1, 0.0.2, 0.0.3, 0.0.4, 0.0.5, 0.0.6, 0.0.7, 0.0.8, 0.0.9, 0.0.10, 0.0.11, 0.0.12, 0.0.13, 0.0.14, 0.0.15, 0.0.16, 0.0.17, 0.0.18, 0.0.19, 0.0.20, 0.0.21, 0.0

In [5]:
!pip install -qU langchain-ollama

In [6]:
!pip install langgraph>0.2.27

## Initialize the Llama Model

In [7]:
from langchain_ollama import ChatOllama

llm = ChatOllama(
    model='llama3.2',
    temperature=0 # ranges from 0 to 1
)

API Reference for initializing the Llama Model
https://python.langchain.com/api_reference/ollama/chat_models/langchain_ollama.chat_models.ChatOllama.html

## Simple Invocation - Prompt and Get Content

In [4]:
from langchain_core.messages import AIMessage

messages = [
    (
        'system',
        'You are a helpful assistant that translates English to Spanish. Translate the inputted sentence.',
    ),
    (
        'human',
        'Goat cheese is delicious.'
    )
]

ai_msg = llm.invoke(messages)
ai_msg

NameError: name 'llm' is not defined

In [11]:
print(ai_msg)

content='El queso de cabra es delicioso.' additional_kwargs={} response_metadata={'model': 'llama3.2', 'created_at': '2025-01-13T21:45:59.641548Z', 'message': {'role': 'assistant', 'content': ''}, 'done_reason': 'stop', 'done': True, 'total_duration': 2322346900, 'load_duration': 1656864900, 'prompt_eval_count': 48, 'prompt_eval_duration': 95000000, 'eval_count': 12, 'eval_duration': 186000000} id='run-84849db6-bbf7-4c16-a64c-66985f33ec67-0' usage_metadata={'input_tokens': 48, 'output_tokens': 12, 'total_tokens': 60}


# Chaining with a prompt template

In [8]:
from langchain_core.prompts import ChatPromptTemplate

prompt = ChatPromptTemplate.from_messages(
    [
        (
            "system",
            "You are a helpful assistant that translates {input_language} to {output_language}. Translate the inputted sentence.",
        ),
        (
            'human',
            '{input}',
        )
    ]
)
chain = prompt | llm
ai_msg = chain.invoke(
            {
                'input_language': 'English',
                'output_language': 'French',
                'input': 'The goat cheese is delicious.'
            }
        )

NameError: name 'llm' is not defined

In [5]:
ai_msg.content

'Le fromage de chèvre est délicieux.\n\n(Note: "goat cheese" can also be translated as "fromage de chèvre", which is a more common and idiomatic expression in French.)'

In [6]:
ai_msg.content

'Le fromage de chèvre est délicieux.\n\n(Note: "goat cheese" can also be translated as "fromage de chèvre", which is a more common and idiomatic expression in French.)'

# Testing Ollama to print and run python code for simple addition

Langchain Tool:
Interfaces that an agent, chain, or LLM can use to interact with the world. 
Provides them a function to be able to call (an action to tack)

In [8]:
import os
import traceback
from io import StringIO
from contextlib import redirect_stdout

import langchain
from langchain_ollama import ChatOllama
from langchain.chains import LLMChain
from langchain_core.prompts import PromptTemplate
from langchain.tools import Tool

# Initialize LLM
llm = llm = ChatOllama(
    model='llama3.2',
    temperature=0 # ranges from 0 to 1
)

# Define the Python execution tool (Langchain tool)
def execute_python_code(code:str)->str:
    '''
    Executes the python code provided in a string. Captures output and errors and to return them
    '''
    try:
        # Capture standard output
        f = StringIO()
        with redirect_stdout(f):
            exec(code,{})

        # Return captured output
        return f.getvalue() if f.getvalue() else "Code executed without output."
    except:
        return f'Execution error: {traceback.format_exc()}'

# Define the langchain tool for executing python code
python_executor = Tool(name='PythonExecutor',
                      func=execute_python_code,
                      description='Executes Python code and returns the output or error.'
                      )
# Create the prompt 
prompt_template = """
You are an AI assistant with a Python interpreter. When quantitative analysis or code execution is required for assisting your reasoning, you may generate Python code. 
Generated code should always include necessary imports and provide output relevant to the task.
Start python code using ```python and end it using ```.
If there’s an error during execution, I will return the error message for you to adjust the code and try again.
If you are satisfied with your answer after you have received the code execution results, output "DONE: ANSWER" and provide the answer from the code execution.

User Prompt: {user_prompt}
"""

prompt=PromptTemplate(input_variables=['user_prompt'], template=prompt_template)

# Connect the chain
llm_chain = prompt | llm

Main Dialogue Loop

In [42]:
def interactive_code_execution():
    namespace = {}
    ct = 0
    while ct<2:
        user_input = input('You: ')

        # Exit if user types 'exit'
        if user_input.lower() == 'exit':
            print('Exiting Dialogue.')
            break
        
        # Generate the LLM response with potential code to execute
        response = llm_chain.invoke({'user_prompt': user_input})
        print('\nLLM Response: ',response.content, '\n')

        # Check for Python code blocks in the LLM response
        if "```python" in response.content:
            code_snippet = response.content.split("```python")[1].split("```")[0].strip()
            print('\nExecuting code snippet:\n', code_snippet)
            try:
                # Execute the code
                execution_result = python_executor.run(code_snippet)
    
                # Append execution results to continue the conversation
                print('\nExecution Result:', execution_result)
                result_context = (f"User asked: '{user_input}'\n"
                        f"Your response: '{response.content}'\n"
                        f"Your Code Execution Result: {execution_result}\n"   
                )
                
                # Pass the result back as context to refine or complete the response
                response = llm_chain.invoke({'user_prompt': result_context})
                print('\nLLM Response after execution:', response.content, '\n')
                if 'DONE' in response.content:
                    break
            except Exception as e:
                print(f'\nError during code execution: {e}')
        else:
            print('No code detected, continuing dialogue.\n')
        ct+=1

if __name__ == "__main__":
    interactive_code_execution()    

You:  decorate a room with multiple objects. each object should have random x,y coordinates between 0,0 and 100,100. Find the 2 objects that are closest to each other. make 10 objects



LLM Response:  ```python
import random
import math

# Function to generate a random point within the given bounds
def generate_random_point():
    return (random.randint(0, 100), random.randint(0, 100))

# List to store all objects
objects = []

# Generate and add 10 objects with random coordinates
for _ in range(10):
    obj = {'x': generate_random_point()[0], 'y': generate_random_point()[1]}
    objects.append(obj)

# Function to calculate the distance between two points
def calculate_distance(point1, point2):
    return math.sqrt((point1['x'] - point2['x'])**2 + (point1['y'] - point2['y'])**2)

# Initialize minimum distance and closest pair of objects
min_distance = float('inf')
closest_pair = None

# Iterate over all pairs of objects to find the closest pair
for i in range(len(objects)):
    for j in range(i+1, len(objects)):
        dist = calculate_distance(objects[i], objects[j])
        if dist < min_distance:
            min_distance = dist
            closest_pair = (objects

You:  ok continue



LLM Response:  I'm ready to assist. What's the first prompt or question? 

No code detected, continuing dialogue.



# Hackathon Dev

Changing to chatbot functions

In [8]:
import yaml

def load_yaml_config(file_path):
    with open(file_path, 'r') as file:
        config = yaml.safe_load(file)
    return config

llm_config_path = "llm_config.yaml"
llm_config = load_yaml_config(llm_config_path)

default_system_prompt = """
You are an AI assistant with a Python interpreter. When quantitative analysis or code execution is required for assisting your reasoning, you may generate Python code. 
Generated code should always include necessary imports and provide output relevant to the task.
If code is generated, make sure it begins with ```python and it ends using ```.
If there’s an error during execution, I will return the error message for you to adjust the code and try again.
After you generate code, I will run it and input it into the next query I submit to you. At which point, 
if you are satisfied with your answer after, and only after, you have received the code execution results, output "DONE: ANSWER" and provide the answer from the code execution.
"""

system_prompt = llm_config.get('system_prompt', default_system_prompt)
model_name = llm_config.get('model', 'llama3.2')
temperature = llm_config.get('temperature')
filter_system_prompt = llm_config.get('filter_system_prompt')

In [9]:
import os
import traceback
from io import StringIO
from contextlib import redirect_stdout

import langchain
from langchain_ollama import ChatOllama
from langchain.chains import LLMChain
from langchain.tools import Tool
from langchain_core.prompts import ChatPromptTemplate, MessagesPlaceholder
from langchain_core.messages import HumanMessage, AIMessage, SystemMessage



# Initialize LLM
model = ChatOllama(
    model=model_name,
    temperature=temperature # ranges from 0 to 1
)


In [10]:
filter_model = ChatOllama(
    model=model_name,
    temperature=temperature # ranges from 0 to 1
)


In [11]:
# Define the Python execution tool (Langchain tool)
def execute_python_code(code:str)->str:
    '''
    Executes the python code provided in a string. Captures output and errors and to return them
    '''
    try:
        # Capture standard output
        f = StringIO()
        with redirect_stdout(f):
            exec(code,{})

        # Return captured output
        return f.getvalue() if f.getvalue() else "Code executed without output."
    except:
        return f'Execution error: {traceback.format_exc()}'

# Define the langchain tool for executing python code
python_executor = Tool(name='PythonExecutor',
                      func=execute_python_code,
                      description='Executes Python code and returns the output or error.'
                      )

prompt = ChatPromptTemplate.from_messages(
    [
        (
            "system", 
            f"{system_prompt}",
        ),
        MessagesPlaceholder(variable_name='messages'),
    ]
)

In [12]:
filter_prompt = ChatPromptTemplate.from_messages(
    [
        (
            "system", 
            f"{filter_system_prompt}",
        ),
        MessagesPlaceholder(variable_name='messages'),
    ]
)

In [13]:
from langgraph.checkpoint.memory import MemorySaver
from langgraph.graph import START, MessagesState, StateGraph

# Define a new graph
workflow = StateGraph(state_schema=MessagesState)

# Define the function that calls the model
def call_model(state: MessagesState):
    chain = prompt | model
    response = chain.invoke(state)
    return {"messages": response}

# Define the (single) node in the graph
workflow.add_edge(START, "model")
workflow.add_node("model",call_model)

# Add memory
memory = MemorySaver()
app = workflow.compile(checkpointer=memory)

In [14]:
# Define a new graph
filter_workflow = StateGraph(state_schema=MessagesState)

# Define the function that calls the model
def call_filter_model(state: MessagesState):
    chain = filter_prompt | filter_model
    response = chain.invoke(state)
    return {"messages": response}

# Define the (single) node in the graph
filter_workflow.add_edge(START, "model")
filter_workflow.add_node("model",call_filter_model)

# Add memory
filter_memory = MemorySaver()
filter_app = filter_workflow.compile(checkpointer=filter_memory)

Below config must be passed into the runnable every time (stores the conversation)

In [15]:
config = {"configurable": {"thread_id": "abc129"}}

## Initial Loop
Has the problem of requiring user input after running a python code given by the LLM that did not work

In [9]:
while True:
        query = input('You: ')
        # Exit if user types 'exit'
        if query.lower() == 'exit':
            print('Exiting Dialogue.')
            break
        input_messages = [HumanMessage(query)]
        response = app.invoke({'messages': input_messages},config)
        print("\n\nResponse:",response['messages'][-1].content,'\n')

        # Check for Python code blocks in the LLM response
        if "```python" in response['messages'][-1].content:
            code_snippet = response['messages'][-1].content.split("```python")[1].split("```")[0].strip()
            print('\nExecuting code snippet:\n', code_snippet)
            try:
                # Execute the code
                execution_result = python_executor.run(code_snippet)
    
                # Append execution results to continue the conversation
                print('\nExecution Result:', execution_result)
                result_context = (#f"User asked: '{user_input}'\n"
                        #f"Your response: '{response.content}'\n"
                        f"Your Code Execution Result: {execution_result}\n"   
                )
                
                # Pass the result back as context to refine or complete the response
                response = app.invoke({'messages': result_context},config)
                print('\nLLM Response after execution:', response['messages'][-1].content, '\n')
                if 'DONE' in response['messages'][-1].content:
                    break
            except Exception as e:
                print(f'\nError during code execution: {e}')
        else:
            print('No code detected, continuing dialogue.\n')

You:  hardcode the XY positions of 10 random objects. Then find the two objects that are closest together




Response: ```python
import math
import random

# Hardcode the XY positions of 10 random objects
objects = [(random.uniform(0, 100), random.uniform(0, 100)) for _ in range(10)]

# Function to calculate distance between two points
def distance(point1, point2):
    return math.sqrt((point1[0] - point2[0])**2 + (point1[1] - point2[1])**2)

# Find the minimum distance and the corresponding objects
min_distance = float('inf')
closest_objects = None

for i in range(len(objects)):
    for j in range(i+1, len(objects)):
        dist = distance(objects[i], objects[j])
        if dist < min_distance:
            min_distance = dist
            closest_objects = (i, j)

# Print the results
print("XY positions of 10 random objects:")
for i, obj in enumerate(objects):
    print(f"Object {i+1}: ({obj[0]}, {obj[1]})")

print("\nThe two objects that are closest together are:")
print(f"Objects {closest_objects[0]+1} and {closest_objects[1]+1}")
print(f"Their distance is: {min_distance:.2f}")

# Print 

## Improved Old Loop - catches and loops if python error occurs from LLM-generated code

In [ ]:
retries = 0
max_retries = 3

while True:
        query = input('You: ')
        # Exit if user types 'exit'
        if query.lower() == 'exit':
            print('Exiting Dialogue.')
            break
            
        input_messages = [HumanMessage(query)]
        response = app.invoke({'messages': input_messages},config)
        print("\n\nResponse:",response['messages'][-1].content,'\n')

        # Check for Python code blocks in the LLM response
        if "```python" in response['messages'][-1].content:
            code_snippet = response['messages'][-1].content.split("```python")[1].split("```")[0].strip()

            while retries < max_retries:
                print('\nExecuting code snippet:\n', code_snippet)
                try:
                    # Execute the code
                    execution_result = python_executor.run(code_snippet)
        
                    # Append execution results to continue the conversation
                    print('\nExecution Result:', execution_result)
                    result_context = (f"Your Code Execution Result: {execution_result}\n")
                    
                    # Pass the result back as context to refine or complete the response
                    response = app.invoke({'messages': [HumanMessage(result_context)]}, config)
                    print('\nLLM Response after execution:', response['messages'][-1].content, '\n')

                    # Update the code snippet if the LLM provided refined code
                    if "```python" in response['messages'][-1].content:
                        code_snippet = response['messages'][-1].content.split("```python")[1].split("```")[0].strip()
                    else:
                        print('No refined code provided, exiting refinement loop.')
                        break
                        
                except Exception as e:
                    retries += 1
                    # Provide erro to LLM for feedback
                    print(f'\nError during code execution (Attempt {retries}/{max_retries}): {e}')
                    error_context = f"Error during code execution: {str(e)}"
                    response = app.invoke({'messages': [HumanMessage(error_context)]},config)
                    print('\nLLM Response after error: ', response['messages'][-1].content, '\n')
                    
                    # Update the code snippet if the LLM provided refined code
                    if "```python" in response['messages'][-1].content:
                        code_snippet = response['messages'][-1].content.split("```python")[1].split("```")[0].strip()
                    else:
                        print('No refined code provided, exiting refinement loop.')
                        break
                        
            if 'DONE' in response['messages'][-1].content:
                break
            if retries == max_retries:
                print('Max retries reached, exiting code execution loop.')    
        else:
            print('No code detected, continuing dialogue.\n')

hardcode 10 random household objects and their XY positions. Do not use random functions. Then, find the two objects that are closest together

# LLM_Solver Function - Hackathon

In [1]:
from NIM_cluster_llm_script import LLM_Solver
import os
import traceback
from io import StringIO
from contextlib import redirect_stdout

import langchain
from langchain_ollama import ChatOllama
from langchain.chains import LLMChain
from langchain.tools import Tool
from langchain_core.prompts import ChatPromptTemplate, MessagesPlaceholder
from langchain_core.messages import HumanMessage, AIMessage, SystemMessage
from langchain_nvidia_ai_endpoints import ChatNVIDIA

ps = LLM_Solver()
string_data = '''
InstanceID class X_location Y_location
1 class:Chair x:0.83293 y:0.94212
2 class:Chair x:0.32390 y:0.83282
3 class:Chair x:0.56789 y:0.12345
4 class:Door x:0.10000 y:0.90000
5 class:Table x:0.11111 y:0.11111
6 class:Lamp x:0.22222 y:0.22222
7 class:Book x:0.33333 y:0.33333
8 class:Plant x:0.44444 y:0.44444
9 class:Cup x:0.55555 y:0.55555
10 class:Shelf x:0.66666 y:0.66666
11 class:Notebook x:0.77777 y:0.77777
12 class:Pen x:0.88888 y:0.88888
13 class:Phone x:0.99999 y:0.99999
14 class:Keyboard x:0.12121 y:0.23232
15 class:Mouse x:0.34343 y:0.45454
16 class:Monitor x:0.56565 y:0.67676
17 class:Speaker x:0.78787 y:0.89898
18 class:Bag x:0.90909 y:0.10101
19 class:Bottle x:0.21212 y:0.32323
20 class:Clock x:0.43434 y:0.54545
'''
ps.run(string_data)

Using thread_id: 1898
You are an AI assistant equipped with a Python interpreter. Your primary goal is to solve 3D Referring Expressions by analyzing the provided sensor data and using Python code as needed. You must generate accurate and concise solutions, ensuring your reasoning is well-documented and logical.

Guidelines:
  1. Solve the Expression: Use the 3D Referring Expression and the accompanying sensor data to compute the required result. Base your solution on spatial relationships, distances, and object properties as specified in the expression.
  2. Python Code Usage:
     - When a solution requires computation, generate Python code to perform the necessary calculations.
     - Include all relevant imports and ensure the code is structured, efficient, and easy to read.
     - Surround code blocks with ```python and ``` to indicate executable Python code.
  3. Iterative Feedback:
     - After generating code, wait for the execution results before providing a final answer.
    

## Testing RegEx Python Code Filtering

In [1]:
from llm_script import LLM_Solver
import os
import traceback
from io import StringIO
from contextlib import redirect_stdout

import langchain
from langchain_ollama import ChatOllama
from langchain.chains import LLMChain
from langchain.tools import Tool
from langchain_core.prompts import ChatPromptTemplate, MessagesPlaceholder
from langchain_core.messages import HumanMessage, AIMessage, SystemMessage

response = {
    "messages": [
        {"content": "Some other text"},
        {"content": '''```python
import math

# Define the coordinates of the door and chairs
door_x, door_y = 0.1, 0.9
chair1_x, chair1_y = 0.83293, 0.94212
chair2_x, chair2_y = 0.32390, 0.83282

# Calculate the Euclidean distance from each chair to the door
distance1 = math.sqrt((chair1_x - door_x) ** 2 + (chair1_y - door_y) ** 2)
distance2 = math.sqrt((chair2_x - door_x) ** 2 + (chair2_y - door_y) ** 2)

# Find the chair with the smallest distance
if distance1 < distance2:
    closest_chair = "Chair 1"
elif distance2 < distance1:
    closest_chair = "Chair 2"
else:
    closest_chair = "Both chairs are equidistant"

print(f"The chair closest to the door is {closest_chair}.")
```'''}
    ]
}
print(response)

{'messages': [{'content': 'Some other text'}, {'content': '```python\nimport math\n\n# Define the coordinates of the door and chairs\ndoor_x, door_y = 0.1, 0.9\nchair1_x, chair1_y = 0.83293, 0.94212\nchair2_x, chair2_y = 0.32390, 0.83282\n\n# Calculate the Euclidean distance from each chair to the door\ndistance1 = math.sqrt((chair1_x - door_x) ** 2 + (chair1_y - door_y) ** 2)\ndistance2 = math.sqrt((chair2_x - door_x) ** 2 + (chair2_y - door_y) ** 2)\n\n# Find the chair with the smallest distance\nif distance1 < distance2:\n    closest_chair = "Chair 1"\nelif distance2 < distance1:\n    closest_chair = "Chair 2"\nelse:\n    closest_chair = "Both chairs are equidistant"\n\nprint(f"The chair closest to the door is {closest_chair}.")\n```'}]}


In [2]:
code_snippet = LLM_Solver.extract_python_code(response)
print(code_snippet)

import math

# Define the coordinates of the door and chairs
door_x, door_y = 0.1, 0.9
chair1_x, chair1_y = 0.83293, 0.94212
chair2_x, chair2_y = 0.32390, 0.83282

# Calculate the Euclidean distance from each chair to the door
distance1 = math.sqrt((chair1_x - door_x) ** 2 + (chair1_y - door_y) ** 2)
distance2 = math.sqrt((chair2_x - door_x) ** 2 + (chair2_y - door_y) ** 2)

# Find the chair with the smallest distance
if distance1 < distance2:
    closest_chair = "Chair 1"
elif distance2 < distance1:
    closest_chair = "Chair 2"
else:
    closest_chair = "Both chairs are equidistant"

print(f"The chair closest to the door is {closest_chair}.")


# Other items to explore
Human in the loop Tool: https://python.langchain.com/docs/how_to/tools_human/

Chatbot vs agent

# Testing LLM Server

In [1]:
server = 'http://localhost:8000/v1'#/chat/completions'

In [2]:
from langchain_nvidia_ai_endpoints import ChatNVIDIA

In [3]:
llm = ChatNVIDIA(base_url=server, model="meta/llama-3.3-70b-instruct")

In [4]:
from langchain_core.messages import AIMessage

messages = [
    (
        'system',
        'You are a helpful assistant that translates English to Spanish. Translate the inputted sentence.',
    ),
    (
        'human',
        '{The goat cheese is terrible.}'
    )
]

ai_msg = llm.invoke(messages)
ai_msg.content

'El queso de cabra es terrible.'

# LLM In the Loop

In [ ]:
system_prompt = """
    You are assisting in an ongoing exploration task where sensor data is continuously being streamed. 
    As the robot explores, the data will be filtered based on the observations and appended to your understanding 
    of the task. You have access to a Python execution tool that will allow you to run Python code when needed to assist 
    in decision-making.

    Instructions:
    - You will receive an initial query that defines the task.
    - As the exploration progresses, you will receive new sensor data, which will be filtered and appended 
      to your understanding of the task.
    - After each new chunk of data, you have two options:
        1. **If you have enough data** to solve the query and decide the task is complete, respond with **"DONE ANSWER"** 
           and provide the final answer to the initial query.
        2. **If more data is needed** or if you cannot yet compute a solution, output **"Continue Exploring"**.
        3. If you believe that running Python code is necessary to refine your understanding or compute the result:
           - Output Python code in the following format:
             ```
             ```python
             <your Python code here>
             ```
             ```
           This code will be executed, and the results will be fed back to you for use or correction.

    - Once the Python code is executed and the results are provided, assess the outcome:
        - If you are satisfied with the results and can now answer the initial query, respond with **"DONE ANSWER"** 
          and provide the final answer.
        - If the results are not sufficient, decide if more adjustments to the code are necessary. You can also output **"Continue Exploring"** 
          to explore more or new code to test out.
    
    Your task is to iteratively process the incoming information, refine your understanding with Python code execution 
    when needed, and either continue exploring or provide the final answer to the initial query once enough data is 
    gathered.
    """

In [ ]:
# Store filtered data as 
self.filtered_data = ''
self.filter_config = {"configurable": {"thread_id": f"{random.randint(1000, 9999)}"}} 
def run(self, input_string):
        config = {"configurable": {"thread_id": f"{self.thread_id}"}}
        
        print('Object Info received')
        # Exit if user types 'exit'
        if self.expression_query.lower() == 'exit':
            print('Exiting Dialogue.')
            return
        
        input_messages = [HumanMessage(self.expression_query+'\n'+self.filter_prompt+'\n Here is the sensor data to use to solve the 3D referring expression:'+input_string)]
        print(f"You: {input_messages[-1].content}")
        
        response = self.filter_app.invoke({'messages': input_messages},self.filter_config)
        print("\n\nResponse:",response['messages'][-1].content,'\n')
        self.filtered_data = response['messages'][-1].content # changed filtered_string to self.filtered_data
        
        input_messages = [HumanMessage(self.expression_query+'\nFiltered Data Observed:\n'+filtered_string+'\nNow you can output Python code to calculate your response to assist your decision-making.')]
        print(f"You: {input_messages[-1].content}")

        response = self.app.invoke({'messages': input_messages},config)
        print("\n\nResponse:",response['messages'][-1].content,'\n')

        # Check for Python code blocks in the LLM response
        if "```python" in response['messages'][-1].content:
            code_snippet = response['messages'][-1].content.split("```python")[1].split("```")[0].strip()
            retries = 0
            while retries < self.max_retries:
                print('\nExecuting code snippet:\n', code_snippet)
                try:
                    # Execute the code
                    execution_result = self.python_executor.run(code_snippet)
        
                    # Append execution results to continue the conversation
                    print('\nExecution Result:', execution_result)
                    result_context = (f"Your Code Execution Result: {execution_result}\n")
                    
                    # Pass the result back as context to refine or complete the response
                    response = self.app.invoke({'messages': [HumanMessage(result_context)]}, config)
                    print('\nLLM Response after execution:', response['messages'][-1].content, '\n')

                    # Update the code snippet if the LLM provided refined code
                    if "```python" in response['messages'][-1].content:
                        code_snippet = response['messages'][-1].content.split("```python")[1].split("```")[0].strip()
                    elif: "Continue Exploring" in response['messages'][-1].content:
                        return 'Continue Exploring'
                    elif: "DONE ANSWER" in response['messages'][-1].content:
                        return response['messages'][-1].content
                    else:
                        print('No refined code provided, exiting refinement loop.')
                        break
                        
                except Exception as e:
                    retries += 1
                    # Provide erro to LLM for feedback
                    print(f'\nError during code execution (Attempt {retries}/{self.max_retries}): {e}')
                    error_context = f"Error during code execution: {str(e)}"
                    response = self.app.invoke({'messages': [HumanMessage(error_context)]},config)
                    print('\nLLM Response after error: ', response['messages'][-1].content, '\n')
                    
                    # Update the code snippet if the LLM provided refined code
                    if "```python" in response['messages'][-1].content:
                        code_snippet = response['messages'][-1].content.split("```python")[1].split("```")[0].strip()
                    elif: "Continue Exploring" in response['messages'][-1].content:
                        return 'Continue Exploring'
                    elif: "DONE ANSWER" in response['messages'][-1].content:
                        return response['messages'][-1].content
                    else:
                        print('No refined code provided, exiting refinement loop.')
                        break
                        
            if retries == self.max_retries:
                print('Max retries reached, exiting code execution loop.')    
                return 'Continue Exploring'

        elif: "Continue Exploring" in response['messages'][-1].content:
            return 'Continue Exploring'
    
        else:
            print('No code detected, ending dialogue.\n')